# Experiment H2 · Supervised fine-tuning of DINOv2 ViT-S/14

Restore F0 seed 2026, verify its validation performance, then train only the final two
transformer blocks, final LayerNorm, and original classification head on training images.
No cached training features, TTA, or ensemble are used. All results below are computed
when you execute this notebook; this distributed notebook contains no experiment results.

**Colab:** choose **Runtime → Change runtime type → T4 GPU**, edit section 1, and run
all cells in order. Supply the original `dinov2_p3_e4_aug123_head2026.pt` before running.
Training on CPU is supported but slow. Each run creates a new output folder.

Source inspected: [F0/H0/H1 notebook](https://github.com/dohyunii/Car-Classification-Benchmark/blob/5a393fa56ae33a183a2268253528a795b5a450fe/Baseline_Submission.ipynb).
Its sequential head keys, P3 geometry transforms, saved feature normalization, Kaggle
competition slug, and stem-based image IDs are preserved. The workspace's older
DINOv1 notebook is left untouched. F0's four cached augmented views are replaced by
fresh online augmentation each epoch, as required for supervised backbone updates.

## 0. Environment and imports
Use Colab's installed PyTorch/TorchVision (PyTorch ≥ 2.4 for `torch.amp`). Do not
upgrade them independently. DINOv2 can use its standard PyTorch attention fallback
without xFormers. The hub source is pinned below; original pretrained weights are
downloaded automatically. Internet access is needed for setup and Kaggle download.

In [ ]:
%pip -q install "timm==1.0.22" "scikit-learn==1.7.2" "pandas==2.2.3" "Pillow==11.3.0" "tqdm==4.67.1" "matplotlib==3.10.7" "kagglehub==1.0.3"

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import random
import uuid
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from IPython.display import display

assert hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler'), 'Use PyTorch >= 2.4.'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'
print('PyTorch:', torch.__version__, '| TorchVision:', torchvision.__version__, '| Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    warnings.warn('CPU execution is supported; full fine-tuning will be slow.')

## 1. Configuration and reproducibility
For Drive persistence, set `USE_DRIVE=True`, put the checkpoint in
`MyDrive/Car-Classification-Benchmark/student_outputs/`, and edit paths if needed.
Otherwise upload the checkpoint with Colab's Files panel into `/content/student_outputs/`.
Outputs in `/content/` disappear when the runtime ends; download them or use Drive.

The existing competition slug is retained. With `DOWNLOAD_FROM_KAGGLE=True`, accept
the competition rules on Kaggle and set Colab Secret `KAGGLE_API_TOKEN` with notebook
access (or an existing environment credential). To use an extracted/uploaded dataset,
set it to `False` and set `DATA_ROOT` to the folder containing `train/`, `val/`,
`test/`, and `sample_submission.csv`. No developer-local files are needed.

H2 randomness is seed 2026; seed 123 identifies the F0 augmentation checkpoint only.
Deterministic settings and worker seeds improve repeatability; exact results can still
depend on GPU/software versions, which are recorded with the run.

In [ ]:
USE_DRIVE = False
DOWNLOAD_FROM_KAGGLE = True
COMPETITION = 'idnn-swe-3052-assignment-3-flowers-classification'
DATA_ROOT = Path('/content/car_data')
PERSISTENT_ROOT = (Path('/content/drive/MyDrive/Car-Classification-Benchmark')
                   if USE_DRIVE else Path('/content'))
F0_CHECKPOINT = PERSISTENT_ROOT / 'student_outputs/dinov2_p3_e4_aug123_head2026.pt'
OUTPUT_ROOT = PERSISTENT_ROOT / 'student_outputs_H2'

SEED = 2026
SEED_AUG_F0 = 123
SEED_HEAD_F0 = 2026
MODEL_NAME = 'dinov2_vits14'
HUB_REF = '7764ea0f912e53c92e82eb78a2a1631e92725fc8'
FEATURE_DIM = 384
NUM_CLASSES = 20
IMAGE_SIZE = 224
BATCH_SIZE = 8
NUM_WORKERS = 2  # Set 0 if Colab worker processes fail; worker seeds remain controlled.
MAX_EPOCHS = 10
PATIENCE = 3
BACKBONE_LR = 2e-6
HEAD_LR = 3e-5
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0
EXPECTED_BASELINE_ACC = 0.98
EXPECTED_BASELINE_F1 = 0.9803
BASELINE_F1_ATOL = 5e-4  # Published F1 is approximate; accuracy must match exactly.
PLOT_CONFUSION_MATRIX = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=True)

seed_everything(SEED)
if not F0_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f'Missing F0 checkpoint: {F0_CHECKPOINT}. Upload the original '
        'dinov2_p3_e4_aug123_head2026.pt through Colab Files or mount Drive, '
        'then edit F0_CHECKPOINT. H2 cannot substitute random head weights.')

if DOWNLOAD_FROM_KAGGLE:
    import kagglehub
    if not os.environ.get('KAGGLE_API_TOKEN'):
        try:
            from google.colab import userdata
            os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
        except Exception as exc:
            raise RuntimeError('Enable Colab Secret KAGGLE_API_TOKEN, or disable Kaggle download '
                               'and provide extracted data at DATA_ROOT.') from exc
    DATA_ROOT = Path(kagglehub.competition_download(COMPETITION))

for name in ('train', 'val', 'test'):
    assert (DATA_ROOT / name).is_dir(), f'Missing {DATA_ROOT / name}; edit DATA_ROOT.'
assert (DATA_ROOT / 'sample_submission.csv').is_file(), 'Original sample_submission.csv is required.'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
BEST_PATH = OUTPUT_DIR / 'H2_dinov2_last2_seed2026.pt'
assert BEST_PATH.resolve() != F0_CHECKPOINT.resolve()
print('Data:', DATA_ROOT, '\nF0:', F0_CHECKPOINT, '\nH2 outputs:', OUTPUT_DIR)

## 2. Dataset loading and image preprocessing
Use the exact F0 BICUBIC geometry and ImageNet normalization. Training reads only
`train/`; validation is used for evaluation and checkpoint selection. Test labels
are never loaded. Class-folder prefixes must agree with Kaggle's integer labels.

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1),
                                 interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
PREPROCESSING = {
    'train': {'random_resized_crop': IMAGE_SIZE, 'scale': [0.8, 1.0],
              'ratio': [0.9, 1.1], 'horizontal_flip_p': 0.5},
    'eval': {'resize_short_edge': 256, 'center_crop': IMAGE_SIZE},
    'interpolation': 'BICUBIC', 'mean': MEAN, 'std': STD,
    'train_repr': repr(train_transform), 'eval_repr': repr(eval_transform),
    'feature_normalization': 'fixed F0 checkpoint mean/std; no recomputation',
}
train_ds = datasets.ImageFolder(DATA_ROOT / 'train', transform=train_transform)
val_ds = datasets.ImageFolder(DATA_ROOT / 'val', transform=eval_transform)
assert train_ds.class_to_idx == val_ds.class_to_idx, 'Train/validation class mappings differ.'
assert len(train_ds.classes) == NUM_CLASSES
assert (len(train_ds), len(val_ds)) == (700, 200), 'Expected the original 700/200 split for F0 reproduction.'
for i, folder in enumerate(train_ds.classes):
    assert folder.startswith(f'{i:03d}_'), f'Class-label mismatch: {folder}'
assert not ({Path(p).resolve() for p, _ in train_ds.samples}
            & {Path(p).resolve() for p, _ in val_ds.samples}), 'Train/validation paths overlap.'
print('Training:', len(train_ds), '| Validation:', len(val_ds))
print('Class mapping:', train_ds.class_to_idx)

## 3. Test image discovery and Kaggle ID validation
Require the original sample file and preserve its string IDs and row order, including
leading zeros. As in the existing notebook, each flat test filename stem is its ID.
Missing, extra, empty, or duplicate IDs cause an error before training.

In [ ]:
def discover_test_images(test_dir, sample_path):
    extensions = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
    files = sorted(p for p in test_dir.iterdir() if p.is_file() and p.suffix.lower() in extensions)
    assert files, f'No test images in {test_dir}'
    sample = pd.read_csv(sample_path, dtype={'image_id': str}, keep_default_na=False)
    assert list(sample.columns) == ['image_id', 'label'], 'Expected image_id,label columns.'
    ids = sample.image_id.tolist()
    assert ids and all(isinstance(i, str) and i.strip() for i in ids), 'Empty image IDs.'
    assert len(ids) == len(set(ids)), 'Duplicate sample image IDs.'
    by_id = {p.stem: p for p in files}
    assert len(by_id) == len(files), 'Two test filenames share the same stem.'
    missing, extra = set(ids) - set(by_id), set(by_id) - set(ids)
    assert not missing and not extra, f'Test ID mismatch: missing={sorted(missing)[:10]}, extra={sorted(extra)[:10]}'
    return sample, ids, by_id

class TestImages(Dataset):
    def __init__(self, ids, by_id, transform):
        self.ids, self.by_id, self.transform = list(ids), by_id, transform
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, index):
        with Image.open(self.by_id[self.ids[index]]) as im:
            return self.transform(im.convert('RGB'))

sample_path = DATA_ROOT / 'sample_submission.csv'
sample, required_ids, by_id = discover_test_images(DATA_ROOT / 'test', sample_path)
test_ds = TestImages(required_ids, by_id, eval_transform)
print('Test images:', len(test_ds), '| First sample IDs:', required_ids[:3])

## 4. Load pretrained DINOv2 and restore the best F0 classification head
Load only the supplied checkpoint; validate all expected metadata and strict head
state keys/shapes. Normalization vectors are persistent, non-trainable model buffers.
They are used exactly as saved, without refitting or clamping. `backbone(images)` is
the same CLS representation as F0. The head architecture and sequential keys match
the inspected source. A pinned DINOv2 source revision loads its original pretrained weights.

In [ ]:
HEAD_EXPERIMENT = 'E4_mlp_bn_drop03'
HEAD_CONFIG = {'hidden_dim': 128, 'use_bn': True, 'dropout': 0.3}

def load_f0_checkpoint(path, class_to_idx):
    if not path.is_file():
        raise FileNotFoundError(f'Provide the original F0 checkpoint at {path}.')
    try:
        checkpoint = torch.load(path, map_location='cpu', weights_only=True)
    except Exception as exc:
        raise RuntimeError('F0 checkpoint could not be safely loaded. Use the original torch.save '
                           'dictionary containing tensors and primitive metadata; no random fallback.') from exc
    required = {'head_state_dict', 'feature_mean', 'feature_std', 'class_to_idx',
                'backbone', 'preprocessing', 'head_experiment', 'head_config',
                'seed_aug', 'seed_head', 'best_accuracy'}
    assert isinstance(checkpoint, dict) and required <= checkpoint.keys(), 'Incomplete F0 checkpoint.'
    expected = {'backbone': MODEL_NAME, 'preprocessing': 'P3_geometry_multiview',
                'head_experiment': HEAD_EXPERIMENT, 'head_config': HEAD_CONFIG,
                'seed_aug': SEED_AUG_F0, 'seed_head': SEED_HEAD_F0,
                'class_to_idx': class_to_idx}
    for key, value in expected.items():
        assert checkpoint[key] == value, f'F0 metadata mismatch for {key}: {checkpoint[key]!r}'
    for key in ('feature_mean', 'feature_std'):
        vector = checkpoint[key]
        assert torch.is_tensor(vector) and vector.numel() == FEATURE_DIM, f'{key} must contain 384 elements.'
        assert vector.is_floating_point() and torch.isfinite(vector).all(), f'Invalid {key}.'
    assert (checkpoint['feature_std'] > 0).all(), 'F0 feature_std must be strictly positive.'
    assert math.isfinite(float(checkpoint['best_accuracy'])), 'Invalid checkpoint accuracy.'
    return checkpoint

def build_f0_head():
    return nn.Sequential(nn.Linear(FEATURE_DIM, 128), nn.BatchNorm1d(128), nn.ReLU(),
                         nn.Dropout(0.3), nn.Linear(128, NUM_CLASSES))

class FineTunedDINO(nn.Module):
    def __init__(self, backbone, head, feature_mean, feature_std):
        super().__init__()
        self.backbone, self.head = backbone, head
        self.register_buffer('feature_mean', feature_mean.detach().reshape(FEATURE_DIM).clone())
        self.register_buffer('feature_std', feature_std.detach().reshape(FEATURE_DIM).clone())
    def forward(self, images):
        features = self.backbone(images).float()
        features = (features - self.feature_mean) / self.feature_std
        return self.head(features)
    def train(self, mode=True):
        super().train(mode)
        # eval() controls stochastic layers, not autograd: final blocks/norm still get gradients.
        self.backbone.eval()
        for module in self.head.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()
        return self

f0 = load_f0_checkpoint(F0_CHECKPOINT, train_ds.class_to_idx)
backbone = torch.hub.load(f'facebookresearch/dinov2:{HUB_REF}', MODEL_NAME,
                          pretrained=True, trust_repo=True, skip_validation=True).to(DEVICE)
assert backbone.embed_dim == FEATURE_DIM and len(backbone.blocks) == 12
head = build_f0_head()
head.load_state_dict(f0['head_state_dict'], strict=True)
model = FineTunedDINO(backbone, head, f0['feature_mean'], f0['feature_std']).to(DEVICE)
model.eval()
print('F0 metadata:', {k: v for k, v in f0.items()
                        if k not in {'head_state_dict', 'feature_mean', 'feature_std'}})
print(model.head)

## 5. Configure trainable and frozen transformer blocks
Freeze everything, then enable only blocks 10–11 and final LayerNorm (zero-based),
plus the restored head. Backbone remains in evaluation mode to preserve pretrained
stochastic-layer behavior; this does **not** disable gradients. Head Dropout is active
during training, but BatchNorm running mean/variance stay fixed because batches of 8
are noisy and the head was already trained. BatchNorm affine weights/bias remain
trainable. This policy is enforced on every `model.train()` call.

In [ ]:
def configure_trainable(model):
    model.backbone.requires_grad_(False)
    for block in model.backbone.blocks[-2:]:
        block.requires_grad_(True)
    model.backbone.norm.requires_grad_(True)
    model.head.requires_grad_(True)
    expected_ids = {id(p) for block in model.backbone.blocks[-2:] for p in block.parameters()}
    expected_ids.update(id(p) for p in model.backbone.norm.parameters())
    actual_ids = {id(p) for p in model.backbone.parameters() if p.requires_grad}
    assert actual_ids == expected_ids, 'Unexpected trainable backbone parameters.'
    return (sum(p.numel() for p in model.backbone.parameters() if p.requires_grad),
            sum(p.numel() for p in model.head.parameters() if p.requires_grad))

TRAINABLE_BACKBONE, TRAINABLE_HEAD = configure_trainable(model)
assert not model.feature_mean.requires_grad and not model.feature_std.requires_grad
print(f'Trainable backbone parameters: {TRAINABLE_BACKBONE:,}')
print(f'Trainable head parameters: {TRAINABLE_HEAD:,}')
print('Trainable blocks:', [10, 11], '| final LayerNorm: trainable')

## 6. Create image DataLoaders
Every epoch loads training images and samples new transforms. No feature tensors are
cached. Validation/test are deterministic and never shuffled. Fixed BatchNorm permits
the final partial batch, so no training images are dropped.

In [ ]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_generator = torch.Generator().manual_seed(SEED)
eval_generator = torch.Generator().manual_seed(SEED + 1)
loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                     pin_memory=AMP_ENABLED, worker_init_fn=seed_worker)
train_loader = DataLoader(train_ds, shuffle=True, drop_last=False,
                          generator=train_generator, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, generator=eval_generator, **loader_kwargs)
test_loader = DataLoader(test_ds, shuffle=False, generator=eval_generator, **loader_kwargs)

## 7. Reproduce F0 baseline performance
Evaluate in FP32 (as in F0) before any optimizer step. Require **98.00%** accuracy and
macro-F1 within 0.0005 of the approximate published **0.9803**. The expected values
are acceptance checks, not H2 results. Failure prints diagnostics and blocks training.
Do not bypass the gate: check the original weights, dataset, transforms, and metadata.

In [ ]:
criterion = nn.CrossEntropyLoss()

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    total_loss, targets, predictions = 0.0, [], []
    for images, labels in tqdm(loader, desc='Validation', leave=False):
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        logits = model(images)  # FP32 evaluation, matching original cached-feature F0 inference.
        total_loss += criterion(logits, labels).item() * labels.size(0)
        targets.extend(labels.cpu().tolist())
        predictions.extend(logits.argmax(1).cpu().tolist())
    return {'val_loss': total_loss / len(targets),
            'val_accuracy': float(accuracy_score(targets, predictions)),
            'val_macro_f1': float(f1_score(targets, predictions, labels=list(range(NUM_CLASSES)),
                                          average='macro', zero_division=0))}, np.asarray(predictions)

def require_baseline(metrics, model, checkpoint):
    ok = (np.isclose(metrics['val_accuracy'], EXPECTED_BASELINE_ACC, rtol=0, atol=1e-8)
          and np.isclose(metrics['val_macro_f1'], EXPECTED_BASELINE_F1, rtol=0, atol=BASELINE_F1_ATOL))
    if not ok:
        diagnostics = {
            'observed': metrics, 'expected_accuracy': EXPECTED_BASELINE_ACC,
            'expected_macro_f1': EXPECTED_BASELINE_F1, 'macro_f1_atol': BASELINE_F1_ATOL,
            'checkpoint_path': str(F0_CHECKPOINT), 'hub_source': HUB_REF,
            'weights': 'original pretrained backbone + strictly restored F0 head',
            'model_training': model.training, 'backbone_training': model.backbone.training,
            'head_training': model.head.training,
            'checkpoint_metadata': {k: v for k, v in checkpoint.items()
                                    if k not in {'head_state_dict', 'feature_mean', 'feature_std'}},
            'dataset_mapping': train_ds.class_to_idx, 'preprocessing': PREPROCESSING,
            'normalization': {k: {'shape': list(getattr(model, k).shape),
                                  'min': float(getattr(model, k).min().item()),
                                  'max': float(getattr(model, k).max().item())}
                              for k in ('feature_mean', 'feature_std')},
        }
        print(json.dumps(diagnostics, indent=2, default=str))
        raise RuntimeError('STOP: F0 baseline did not reproduce. Resolve diagnostics before fine-tuning.')

BASELINE_VERIFIED = False
baseline_metrics, baseline_predictions = evaluate(model, val_loader)
print('Measured F0 baseline:', baseline_metrics)
require_baseline(baseline_metrics, model, f0)
BASELINE_VERIFIED = True
print('PASS: F0 reproduced; supervised fine-tuning is enabled.')

## 8. Configure optimizer and training
Use AdamW parameter groups, constant learning rates, CUDA FP16 autocast/GradScaler,
and unscale before clipping to norm 1.0. CPU uses FP32. There is no classifier warm-up.
Checkpoint ranking is lexicographic `(accuracy, macro-F1)`; a complete baseline model
is saved at epoch 0 before training. Every run has its own directory, protecting
previous checkpoints and submissions. Checkpoints contain the full backbone, head,
normalization buffers, optimizer/scaler state, source identifiers, and configuration.

In [ ]:
assert BASELINE_VERIFIED, 'Run and pass section 7 first.'
optimizer = torch.optim.AdamW([
    {'params': [p for p in model.backbone.parameters() if p.requires_grad],
     'lr': BACKBONE_LR, 'name': 'backbone'},
    {'params': list(model.head.parameters()), 'lr': HEAD_LR, 'name': 'head'},
], weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
trainable_parameters = [p for p in model.parameters() if p.requires_grad]

def metric_rank(metrics):
    return metrics['val_accuracy'], metrics['val_macro_f1']

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def frozen_parameter_digest(model):
    digest = hashlib.sha256()
    for name, parameter in model.backbone.named_parameters():
        if not parameter.requires_grad:
            digest.update(name.encode())
            digest.update(parameter.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

def to_cpu_tree(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {k: to_cpu_tree(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_cpu_tree(v) for v in value]
    if isinstance(value, tuple):
        return tuple(to_cpu_tree(v) for v in value)
    return value

ENVIRONMENT = {'python': platform.python_version(), 'torch': str(torch.__version__),
               'torchvision': str(torchvision.__version__),
               'cuda': str(torch.version.cuda),
               'gpu': torch.cuda.get_device_name(0) if AMP_ENABLED else None,
               'packages': {name: importlib.metadata.version(name) for name in
                            ['timm', 'scikit-learn', 'pandas', 'Pillow', 'tqdm', 'matplotlib', 'kagglehub']}}
OPTIMIZER_SETTINGS = {'name': 'AdamW', 'backbone_lr': BACKBONE_LR, 'head_lr': HEAD_LR,
                      'weight_decay': WEIGHT_DECAY, 'betas': [0.9, 0.999], 'eps': 1e-8,
                      'max_grad_norm': MAX_GRAD_NORM, 'scheduler': None}
RUN_CONFIG = {'seed': SEED, 'batch_size': BATCH_SIZE, 'num_workers': NUM_WORKERS,
              'max_epochs': MAX_EPOCHS, 'patience': PATIENCE, 'mixed_precision': AMP_ENABLED,
              'image_size': IMAGE_SIZE, 'data_root': str(DATA_ROOT), 'run_id': RUN_ID,
              'train_count': len(train_ds), 'val_count': len(val_ds), 'test_count': len(test_ds)}
F0_SHA256 = file_sha256(F0_CHECKPOINT)
FROZEN_BEFORE = frozen_parameter_digest(model)
BN_BEFORE = {k: v.detach().cpu().clone() for k, v in model.head.state_dict().items()
             if k.endswith(('running_mean', 'running_var', 'num_batches_tracked'))}

def save_checkpoint(path, epoch, metrics):
    payload = {
        'model_state_dict': to_cpu_tree(model.state_dict()),
        'optimizer_state_dict': to_cpu_tree(optimizer.state_dict()),
        'scaler_state_dict': scaler.state_dict(),
        'backbone': MODEL_NAME, 'hub_ref': HUB_REF, 'unfrozen_block_count': 2,
        'unfrozen_block_indices': [10, 11], 'final_layernorm_trainable': True,
        'head_experiment': HEAD_EXPERIMENT, 'head_config': HEAD_CONFIG,
        'preprocessing': PREPROCESSING, 'seed': SEED, 'epoch': epoch,
        'validation_metrics': dict(metrics), 'baseline_metrics': dict(baseline_metrics),
        'class_to_idx': dict(train_ds.class_to_idx), 'optimizer_settings': OPTIMIZER_SETTINGS,
        'run_config': RUN_CONFIG, 'environment': ENVIRONMENT,
        'trainable_backbone_parameters': TRAINABLE_BACKBONE, 'trainable_head_parameters': TRAINABLE_HEAD,
        'f0_checkpoint_name': F0_CHECKPOINT.name, 'f0_checkpoint_sha256': F0_SHA256,
        'f0_metadata': {k: v for k, v in f0.items()
                        if k not in {'head_state_dict', 'feature_mean', 'feature_std'}},
        'backbone_mode': 'eval with autograd', 'head_batchnorm_running_stats': 'fixed',
        'source_notebook_commit': '5a393fa56ae33a183a2268253528a795b5a450fe',
    }
    temporary = path.with_suffix('.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)

best_metrics = dict(baseline_metrics)
best_epoch = 0
history = [{'epoch': 0, 'train_loss': np.nan, **baseline_metrics,
            'backbone_lr': BACKBONE_LR, 'head_lr': HEAD_LR}]
save_checkpoint(BEST_PATH, 0, baseline_metrics)
pd.DataFrame(history).to_csv(OUTPUT_DIR / 'H2_history.csv', index=False)
(OUTPUT_DIR / 'H2_run_config.json').write_text(
    json.dumps({'config': RUN_CONFIG, 'optimizer': OPTIMIZER_SETTINGS,
                'preprocessing': PREPROCESSING, 'environment': ENVIRONMENT,
                'f0_sha256': F0_SHA256}, indent=2))
print('Saved epoch 0 baseline:', BEST_PATH)

## 9. Fine-tune DINOv2
The training forward pass retains autograd. On the first batch verify gradient flow
to both unfrozen blocks, final LayerNorm, and every head parameter (including BN
affine parameters), and absence of frozen-parameter gradients. At the end compare
frozen weights and BN running statistics to their starting values. These checks
detect accidental freezing, optimizer inclusion, or training-mode errors.

In [ ]:
def check_gradient_flow(model):
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            assert parameter.grad is not None, f'Missing gradient: {name}'
        else:
            assert parameter.grad is None, f'Frozen parameter has a gradient: {name}'
    groups = {'block10': model.backbone.blocks[-2], 'block11': model.backbone.blocks[-1],
              'final_norm': model.backbone.norm, 'head': model.head}
    for name, module in groups.items():
        assert any(torch.count_nonzero(p.grad).item() > 0 for p in module.parameters()
                   if p.requires_grad), f'No gradient signal in {name}'
    print('Gradient flow verified for final two blocks, final LayerNorm, and head.')

def train_one_epoch(model, loader, verify_gradients=False):
    model.train()
    assert not model.backbone.training
    assert all(not m.training for m in model.head.modules() if isinstance(m, nn.BatchNorm1d))
    total_loss, seen = 0.0, 0
    for batch_index, (images, labels) in enumerate(tqdm(loader, desc='Train', leave=False)):
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
            logits = model(images)
            loss = criterion(logits, labels)
        if not torch.isfinite(loss):
            raise FloatingPointError('Non-finite training loss; inspect normalization and learning rates.')
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        if verify_gradients and batch_index == 0:
            check_gradient_flow(model)
        torch.nn.utils.clip_grad_norm_(trainable_parameters, MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * labels.size(0)
        seen += labels.size(0)
    assert seen == len(loader.dataset), 'Training images were unexpectedly dropped.'
    return total_loss / seen

assert BASELINE_VERIFIED
# Recheck the gate immediately before training, including on partial notebook reruns.
require_baseline(evaluate(model, val_loader)[0], model, f0)
seed_everything(SEED)
train_generator.manual_seed(SEED)
bad_epochs = 0
for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, verify_gradients=(epoch == 1))
    val_metrics, _ = evaluate(model, val_loader)
    history.append({'epoch': epoch, 'train_loss': train_loss, **val_metrics,
                    'backbone_lr': optimizer.param_groups[0]['lr'],
                    'head_lr': optimizer.param_groups[1]['lr']})
    pd.DataFrame(history).to_csv(OUTPUT_DIR / 'H2_history.csv', index=False)
    print(f'Epoch {epoch:02d} | train loss {train_loss:.4f} | val loss {val_metrics["val_loss"]:.4f} '
          f'| accuracy {val_metrics["val_accuracy"]:.2%} | macro-F1 {val_metrics["val_macro_f1"]:.4f}')
    if metric_rank(val_metrics) > metric_rank(best_metrics):
        best_metrics, best_epoch, bad_epochs = dict(val_metrics), epoch, 0
        save_checkpoint(BEST_PATH, epoch, val_metrics)
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f'Early stopping after {PATIENCE} epochs without a ranking improvement.')
            break

assert frozen_parameter_digest(model) == FROZEN_BEFORE, 'Frozen backbone weights changed.'
for name, original in BN_BEFORE.items():
    assert torch.equal(model.head.state_dict()[name].detach().cpu(), original), f'BN buffer changed: {name}'
assert torch.equal(model.feature_mean.cpu(), f0['feature_mean'].reshape(FEATURE_DIM))
assert torch.equal(model.feature_std.cpu(), f0['feature_std'].reshape(FEATURE_DIM))
best_checkpoint = torch.load(BEST_PATH, map_location='cpu', weights_only=True)
assert best_checkpoint['class_to_idx'] == train_ds.class_to_idx
model.load_state_dict(best_checkpoint['model_state_dict'], strict=True)
best_epoch = best_checkpoint['epoch']
best_metrics = best_checkpoint['validation_metrics']
del best_checkpoint
model.eval()
optimizer.zero_grad(set_to_none=True)
if AMP_ENABLED:
    torch.cuda.empty_cache()
print('Restored best checkpoint; epoch:', best_epoch)

## 10. Evaluate and visualize results
The best checkpoint may still be epoch 0. Accuracy and macro-F1 differences are
reported in percentage points. Epoch 0 has no training loss, so that entry is empty.
All validation points, including epoch 0, are plotted and saved. Validation selection
does not establish a Kaggle improvement; test performance remains unknown.

In [ ]:
final_metrics, final_predictions = evaluate(model, val_loader)
for key in best_metrics:
    assert np.isclose(final_metrics[key], best_metrics[key], rtol=0, atol=1e-6), f'Restore mismatch: {key}'
assert metric_rank(final_metrics) >= metric_rank(baseline_metrics)
comparison = pd.DataFrame([
    {'model': 'F0 baseline', 'epoch': 0, **baseline_metrics},
    {'model': 'H2 selected (epoch 0 allowed)', 'epoch': best_epoch, **final_metrics},
])
display(comparison)
comparison.to_csv(OUTPUT_DIR / 'H2_validation_comparison.csv', index=False)
accuracy_delta_pp = 100 * (final_metrics['val_accuracy'] - baseline_metrics['val_accuracy'])
f1_delta_pp = 100 * (final_metrics['val_macro_f1'] - baseline_metrics['val_macro_f1'])
print(f'Accuracy change: {accuracy_delta_pp:+.2f} percentage points')
print(f'Macro-F1 change: {f1_delta_pp:+.4f} percentage points')
print(f'Best epoch: {best_epoch}; trainable backbone: {TRAINABLE_BACKBONE:,}; head: {TRAINABLE_HEAD:,}')

hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, column, title in zip(axes, ['train_loss', 'val_loss', 'val_accuracy'],
                             ['Training loss', 'Validation loss', 'Validation accuracy (%)']):
    values = hist[column] * (100 if column == 'val_accuracy' else 1)
    ax.plot(hist.epoch, values, marker='o')
    if column != 'train_loss':
        ax.axhline(values.iloc[0], color='gray', linestyle='--', label='Epoch 0 F0')
        ax.legend()
    ax.set(xlabel='Epoch', ylabel=title, title=title)
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'H2_training_curves.png', dpi=160)
plt.show()

report = classification_report(val_ds.targets, final_predictions, labels=list(range(NUM_CLASSES)),
                               target_names=train_ds.classes, zero_division=0, output_dict=True)
pd.DataFrame(report).T.to_csv(OUTPUT_DIR / 'H2_classification_report.csv')
if PLOT_CONFUSION_MATRIX:
    fig, ax = plt.subplots(figsize=(12, 10))
    ConfusionMatrixDisplay.from_predictions(val_ds.targets, final_predictions,
                                           labels=list(range(NUM_CLASSES)), ax=ax, colorbar=False)
    ax.set_title('Selected H2 checkpoint · validation class indices')
    fig.savefig(OUTPUT_DIR / 'H2_confusion_matrix.png', dpi=160, bbox_inches='tight')
    plt.show()

## 11. Generate Kaggle submission
A promising candidate means a fine-tuned epoch (`epoch > 0`) beats epoch 0 by the
same accuracy-first, macro-F1 tie-break ranking. Only then produce a separate H2 CSV.
An accuracy tie with higher F1 is described as such. Otherwise no submission is
generated. Prediction uses a single deterministic image view with no TTA or ensemble.
This cell only writes a CSV; it does not upload it to Kaggle.

In [ ]:
@torch.inference_mode()
def predict_test(model, loader):
    model.eval()
    predictions = []
    for images in tqdm(loader, desc='Test prediction'):
        logits = model(images.to(DEVICE, non_blocking=True))
        predictions.extend(logits.argmax(1).cpu().tolist())
    return np.asarray(predictions, dtype=np.int64)

def build_submission(ids, predictions, sample):
    assert len(predictions) == len(ids) == len(sample)
    assert np.issubdtype(np.asarray(predictions).dtype, np.integer), 'Labels must be integers.'
    submission = pd.DataFrame({'image_id': ids, 'label': predictions})
    assert list(submission.columns) == ['image_id', 'label']
    assert submission.image_id.is_unique
    assert submission.image_id.tolist() == sample.image_id.tolist(), 'Image ID order changed.'
    assert submission.label.between(0, NUM_CLASSES - 1).all()
    return submission

promising_candidate = best_epoch > 0 and metric_rank(final_metrics) > metric_rank(baseline_metrics)
submission_path = None
if promising_candidate:
    test_predictions = predict_test(model, test_loader)
    # Re-read sample to catch accidental changes after discovery.
    original_sample = pd.read_csv(sample_path, dtype={'image_id': str}, keep_default_na=False)
    submission = build_submission(required_ids, test_predictions, original_sample)
    submission_path = OUTPUT_DIR / 'H2_dinov2_last2_seed2026_submission.csv'
    submission.to_csv(submission_path, index=False, mode='x')
    print('Saved candidate:', submission_path, '| Predictions:', len(submission))
    display(submission.head())
else:
    print('No fine-tuned epoch beat the F0 baseline ranking. No H2 submission generated.')

## 12. Experiment summary
Send back `H2_summary.json`, `H2_history.csv`, `H2_validation_comparison.csv`, the
training curves, and any baseline diagnostics. Useful values are baseline/best accuracy
and macro-F1, best epoch, percentage-point differences, and parameter counts. After
manually submitting a candidate, also send its actual Kaggle score for comparison
with F0's reported 0.94693. A validation gain alone is not evidence of a Kaggle gain.
Keep the `.pt` file and run configuration in Drive/downloads for reproducibility;
do not add datasets or model weights to Git.

In [ ]:
if best_epoch == 0:
    conclusion = 'No fine-tuned epoch improved validation ranking; selected model is the original F0 baseline.'
elif accuracy_delta_pp > 0:
    conclusion = 'Fine-tuning improved measured validation accuracy. Kaggle effect is unknown.'
else:
    conclusion = 'Validation accuracy tied F0; macro-F1 improved. Kaggle effect is unknown.'
summary = {'experiment': 'H2', 'baseline': baseline_metrics, 'selected': final_metrics,
           'accuracy_delta_percentage_points': accuracy_delta_pp,
           'macro_f1_delta_percentage_points': f1_delta_pp, 'best_epoch': best_epoch,
           'trainable_backbone_parameters': TRAINABLE_BACKBONE,
           'trainable_head_parameters': TRAINABLE_HEAD,
           'promising_candidate': promising_candidate,
           'submission_path': str(submission_path) if submission_path else None,
           'checkpoint_path': str(BEST_PATH), 'conclusion': conclusion}
(OUTPUT_DIR / 'H2_summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print('Artifacts:', OUTPUT_DIR)
print('Copy/download this run folder before ending a temporary Colab runtime.')